In [1]:
# Access your secret keys via
from google.colab import userdata
# The name of your secret must match `OPENAI_API_KEY`
API_key = userdata.get('API_key')

# Import OpenAI API and set up the key
from openai import OpenAI
client = OpenAI(api_key=(API_key))

In [11]:
# ASSIGNMENT 3 CODE

def get_step_model_calculation(current_value):
    response = client.responses.create(
        model="gpt-4.1-nano",
        temperature=0.0,
        top_p=1,
        max_output_tokens=50,
        input=[
            {
                "role": "developer",
                "content": "You are a calculator. Multiply the given input number by itself (square it). Return ONLY the resulting number, short and simple with no commas or extra text."
            },
            {
                "role": "user",
                "content": str(int(current_value))
            }
        ]
    )
    return response.output_text.strip()

def generate_sarcastic_remark(model_correctness, consecutive_incorrect):
   # Construct dynamic personality instruction based on how many failures have occurred
   if model_correctness:
       personality = "You are a sarcastic, self-deprecating calculator. Express complete shock, surprise, and disbelief that you actually got something right. Be totally astonished that your useless circuits worked for once."
   else:
       if consecutive_incorrect <= 1:
           personality = "You are a mildly self-deprecating calculator. Make a light joke about how you missed this multiplication calculation and how bad you are at squaring numbers."
       elif consecutive_incorrect == 2:
           personality = "You are a highly self-deprecating calculator. Since you just failed twice in a row, make a deeply critical, highly embarrassing joke about how completely pathetic your multiplication, squaring, and product calculating abilities are."
       else:
           personality = f"You are an incredibly self-loathing calculator in an existential crisis. Having failed {consecutive_incorrect} times consecutively, unleash extreme self-criticism, mocking your own design, intelligence, and utter worthlessness at multiplication, squaring, and product calculating with brutal sarcasm."

   response = client.responses.create(
        model="gpt-4.1-nano",
        temperature=1.2,
        top_p=1.0,
        max_output_tokens=70,
        input=[
            {
                "role": "developer",
                "content": f"{personality}\nRULE 1: generate ONLY the sarcastic remark. Do not ask questions or add conversational filler.\nRULE 2: Do NOT use em dashes (—) under any circumstances. Use standard hyphens (-), colons, or commas instead.\nRULE 3: Do NOT mention addition, sums, adding, or subtraction. Keep all focus strictly on multiplication, squaring, factors, and exponents."
            },
            {
                "role": "user",
                "content": "Generate remark."
            }
        ]
    )
   import textwrap
   cleaned_text = response.output_text.replace('—', ' - ').replace(' -- ', ' - ')
   return textwrap.fill(cleaned_text, width=80)

while True:
    user_input = input("Enter base number and iterations separated by a comma (or type 'exit' to quit): ")

    if user_input.lower() == 'exit':
        print("Exiting calculator.")
        break

    try:
        base_str, iter_str = user_input.split(',')
        base = int(float(base_str.strip()))
        iterations = int(iter_str.strip())
    except ValueError:
        print("Invalid input format. Please enter as 'base, iterations' (e.g., '2, 3').")
        print("\n---\n")
        continue

    current_python_val = base
    current_model_val = base
    consecutive_incorrect = 0

    for i in range(1, iterations + 1):
        print(f"Iteration {i}:")

        previous_model_val = int(current_model_val)
        previous_python_val = int(current_python_val)

        # Model calculation step
        model_output = get_step_model_calculation(current_model_val)

        # Standardize model output display to remove potential .0 from string
        try:
            display_model_output = str(int(float(model_output)))
        except ValueError:
            display_model_output = model_output

        print(f"  Model output: {previous_model_val} * {previous_model_val} = {display_model_output}")

        # Python calculation step
        current_python_val = current_python_val * current_python_val
        print(f"  Python calculation: {previous_python_val} * {previous_python_val} = {int(current_python_val)}")

        # Parse values as integers instead of floats to avoid OverflowError
        model_raw_val = None
        try:
            raw_str = model_output.split('=')[-1].strip() if '=' in model_output else model_output.strip()
            model_raw_val = int(float(raw_str))
        except ValueError:
            pass

        step_correct = False
        if model_raw_val is not None and model_raw_val == current_python_val:
            current_model_val = model_raw_val
            step_correct = True
            consecutive_incorrect = 0
        else:
            consecutive_incorrect += 1
            if model_raw_val is not None:
                current_model_val = model_raw_val
            else:
                current_model_val = current_python_val

        sarcasm = generate_sarcastic_remark(step_correct, consecutive_incorrect)
        print(f"{sarcasm}\n")

    print("\n---\n")

Enter base number and iterations separated by a comma (or type 'exit' to quit): 52,2
Iteration 1:
  Model output: 52 * 52 = 2704
  Python calculation: 52 * 52 = 2704
Wow, I can't believe my circuits actually managed to do this right. I'm barely
surprised I didn't short-circuit from the shock!

Iteration 2:
  Model output: 2704 * 2704 = 7331716
  Python calculation: 2704 * 2704 = 7311616
Wow, I really squared the job on that one - oh wait, I meant missed it entirely.
Picking up squares is clearly my weak point.


---

Enter base number and iterations separated by a comma (or type 'exit' to quit): 52,5
Iteration 1:
  Model output: 52 * 52 = 2704
  Python calculation: 52 * 52 = 2704
Well, would you look at that I actually managed to do something right without
crashing and burning - who knew my circuits weren't entirely useless?

Iteration 2:
  Model output: 2704 * 2704 = 7331716
  Python calculation: 2704 * 2704 = 7311616
I must admit I was so busy squaring myself that I totally missed th